# Pipeline OCR & Pre-Processing KUHP

Notebook ini memproses PDF resmi KUHP (UU No. 1 Tahun 2023) menjadi dataset 624 pasal, siap dipakai untuk tahap embedding + FAISS pada chatbot RAG. Alurnya:

1. **Render PDF → gambar** per halaman (dibutuhkan karena GLM-OCR adalah model vision, hanya bisa membaca gambar, bukan PDF langsung).
2. **OCR tiap halaman** dengan GLM-OCR (via Ollama, lokal) → teks mentah.
3. **Gabungkan teks** halaman 1–228 saja (halaman 229 adalah lembar pengesahan, halaman 230–345 adalah bagian Penjelasan yang bukan bagian dari 624 pasal utama).
4. **Cleansing** — buang noise berulang (penanda sambungan halaman, header dokumen, nomor halaman).
5. **Chunking per pasal + metadata** — satu kali pass baca baris demi baris, memecah per `Pasal N` sambil mencap metadata Buku/Bab/Bagian/Paragraf yang sedang berlaku saat itu.
6. **Validasi & debugging anomali** — ketemu beberapa pasal duplikat dan hilang akibat kuirk OCR, ditelusuri akar penyebabnya satu-satu.
7. **Perbaikan tertarget** — tambal nomor pasal spesifik yang bermasalah (bukan regex umum, supaya tidak ikut merusak pasal lain yang sebenarnya sudah benar).
8. **Chunking ulang & validasi final** — hasil akhir: 624 pasal, 0 duplikat, 0 hilang, 0 chunk kosong.
9. **Simpan ke JSON** (`data/processed/chunks.json`) — dataset final, siap untuk tahap embedding.


In [4]:
import fitz  # dari library PyMuPDF
from pathlib import Path
import ollama

In [5]:
PDF_PATH = "data/UU_Nomor_1_Tahun_2023.pdf"      
IMAGES_DIR = Path("data/pages")  # folder tempat nyimpen hasil gambar per halaman
DPI = 250                   # resolusi gambar, makin tinggi makin jelas tapi makin berat

In [21]:
# IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# doc = fitz.open(PDF_PATH)
# zoom = DPI / 72
# matrix = fitz.Matrix(zoom, zoom)

# page = doc.load_page(0)  # halaman pertama, index mulai dari 0
# pix = page.get_pixmap(matrix=matrix)
# pix.save(str(IMAGES_DIR / "page_0001.png"))

# print("Berhasil, cek folder 'pages'")

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

doc = fitz.open(PDF_PATH)
zoom = DPI / 72
matrix = fitz.Matrix(zoom, zoom)

image_paths = []
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    pix = page.get_pixmap(matrix=matrix)
    img_path = IMAGES_DIR / f"page_{page_num + 1:04d}.png"
    pix.save(str(img_path))
    image_paths.append(img_path)
    
    if (page_num + 1) % 10 == 0 or page_num == len(doc) - 1:
        print(f"[render] halaman {page_num + 1}/{len(doc)}")

doc.close()
print(f"\nSelesai. Total {len(image_paths)} halaman berhasil dirender.")

[render] halaman 10/345
[render] halaman 20/345
[render] halaman 30/345
[render] halaman 40/345
[render] halaman 50/345
[render] halaman 60/345
[render] halaman 70/345
[render] halaman 80/345
[render] halaman 90/345
[render] halaman 100/345
[render] halaman 110/345
[render] halaman 120/345
[render] halaman 130/345
[render] halaman 140/345
[render] halaman 150/345
[render] halaman 160/345
[render] halaman 170/345
[render] halaman 180/345
[render] halaman 190/345
[render] halaman 200/345
[render] halaman 210/345
[render] halaman 220/345
[render] halaman 230/345
[render] halaman 240/345
[render] halaman 250/345
[render] halaman 260/345
[render] halaman 270/345
[render] halaman 280/345
[render] halaman 290/345
[render] halaman 300/345
[render] halaman 310/345
[render] halaman 320/345
[render] halaman 330/345
[render] halaman 340/345
[render] halaman 345/345

Selesai. Total 345 halaman berhasil dirender.


In [22]:
print(f"Jumlah gambar yang dihasilkan: {len(image_paths)}")

Jumlah gambar yang dihasilkan: 345


## Sanity check: render 1 halaman dulu

Sebelum me-render seluruh 345 halaman (yang butuh waktu), kita tes dulu 1 halaman untuk memastikan `fitz` (PyMuPDF) bisa membuka PDF dan hasil gambarnya jelas terbaca. Kode percobaan 1-halaman ini sudah di-comment-out di sel render di bawah — sel yang aktif langsung me-render semua halaman sekaligus setelah sanity check-nya lolos.


## Sanity check: coba OCR 1 halaman dulu

Setelah `page_0001.png` berhasil dibuat, kita tes kirim gambar itu ke GLM-OCR (via Ollama) untuk memastikan modelnya bisa membaca teks dengan baik dan prompt-nya sudah tepat, sebelum dipakai untuk OCR seluruh halaman.

Catatan: `num_ctx` dinaikkan ke 8192 (dari default 4096) karena gambar hasil render (DPI 250) butuh lebih banyak token konteks daripada default Ollama, dan sempat error `exceed_context_size_error` kalau dibiarkan default.


In [20]:
prompt = (
    "Ekstrak seluruh teks pada gambar ini secara verbatim (apa adanya), "
    "tanpa menambahkan, memparafrase, atau memperbaiki kata apa pun. "
    "Keluarkan hanya teks isi dokumen."
)

response = ollama.chat(
    model="glm-ocr",
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": [str(IMAGES_DIR / "page_0001.png")]
        }
    ],
    options={
        "temperature": 0,
        "num_ctx": 8192   # naikkan dari default 4096
    }
)

hasil_teks = response["message"]["content"]
print(hasil_teks)

Undang-Undang Republik Indonesia

NOMOR 1 TAHUN 2023

TENTANG

KITAB Undang-Undang HUKUM PIDANA

DENGAN RAHMAT TUHAN YANG MAHA ESA

PRESIDEN REPUBLIK INDONESIA,

Menimbang:

a. bahwa untuk mewujudkan hukum pidana nasional Negara Kesatuan Republik Indonesia yang berdasarkan Pancasila dan Undang-Undang Dasar Negara Republik Indonesia Tahun 1945 serta asas hukum umum yang diakui masyarakat bangsa-bangsa, perlu disusun hukum pidana nasional untuk mengganti Kitab Undang-Undang Hukum Pidana warisan pemerintah kolonial Hindia Belanda;

b. bahwa hukum pidana nasional tersebut harus disesuaikan dengan politik hukum, keadaan, dan perkembangan kehidupan bernasyarakat, berbangsa, dan bernegara yang bertujuan menghormati dan menjunjung tinggi hak asasi manusia, berdasarkan Ketuhanan Yang Maha Esa, kemanusiaan yang adil dan beradab, persatuan Indonesia, kerakyatan yang dipimpin oleh hikmat kebijaksanaan dalam permusyawaratan/perwakilan, dan keadilan sosial bagi seluruh rakyat Indonesia;

c. bahwa ma

In [23]:
def ocr_page(image_path):
    prompt = (
        "Ekstrak seluruh teks pada gambar ini secara verbatim (apa adanya), "
        "tanpa menambahkan, memparafrase, atau memperbaiki kata apa pun. "
        "Keluarkan hanya teks isi dokumen."
    )
    response = ollama.chat(
        model="glm-ocr",
        messages=[
            {
                "role": "user",
                "content": prompt,
                "images": [str(image_path)]
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 8192
        }
    )
    return response["message"]["content"]

## Bungkus jadi fungsi `ocr_page()`

Supaya proses OCR di atas bisa dipakai berulang untuk banyak halaman tanpa menulis ulang kode `ollama.chat(...)`, logikanya dibungkus jadi fungsi `ocr_page(image_path)`. Fungsi ini baru benar-benar memproses sesuatu saat *dipanggil* (misal `ocr_page(image_paths[0])`) — mendefinisikannya saja (sel di atas) tidak memicu request apa pun ke Ollama.


## Tahap 2: OCR semua halaman (dengan checkpoint)

Loop `ocr_page()` ke seluruh 345 halaman. Karena prosesnya bisa makan waktu lama (~1–3 jam) dan berisiko terhenti di tengah jalan (laptop sleep, error jaringan, dll), hasil disimpan bertahap tiap 10 halaman ke file **`data/ocr_raw/ocr_checkpoint.json`** — supaya kalau proses terputus, progres yang sudah selesai tidak hilang.

**Catatan penting:** sel di bawah ini hanya perlu dijalankan **sekali** saat OCR pertama kali dilakukan. Untuk sesi kerja berikutnya (setelah kernel restart, dll), cukup **load ulang** hasil yang sudah tersimpan di `ocr_checkpoint.json` — lihat 2 sel setelah ini — tanpa perlu mengulang OCR dari awal.


In [10]:
import json
from pathlib import Path

In [ ]:


CHECKPOINT_DIR = Path("data/ocr_raw")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "ocr_checkpoint.json"

page_results = []

for i, gambar in enumerate(image_paths, start=1):
    try:
        teks = ocr_page(gambar)
    except Exception as e:
        print(f"[GAGAL] halaman {i}: {e}")
        teks = ""
    
    page_results.append({"page": i, "text": teks})
    
    if i % 10 == 0 or i == len(image_paths):
        print(f"[ocr] {i}/{len(image_paths)} halaman selesai")
        # simpan checkpoint tiap 10 halaman, biar kalau terhenti tidak hilang semua
        with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump(page_results, f, ensure_ascii=False, indent=2)

print("\nSelesai OCR semua halaman.")

[ocr] 10/345 halaman selesai
[ocr] 20/345 halaman selesai
[ocr] 30/345 halaman selesai
[ocr] 40/345 halaman selesai
[ocr] 50/345 halaman selesai
[ocr] 60/345 halaman selesai
[ocr] 70/345 halaman selesai
[ocr] 80/345 halaman selesai
[ocr] 90/345 halaman selesai
[ocr] 100/345 halaman selesai
[ocr] 110/345 halaman selesai
[ocr] 120/345 halaman selesai
[ocr] 130/345 halaman selesai
[ocr] 140/345 halaman selesai
[ocr] 150/345 halaman selesai
[ocr] 160/345 halaman selesai
[ocr] 170/345 halaman selesai
[ocr] 180/345 halaman selesai
[ocr] 190/345 halaman selesai
[ocr] 200/345 halaman selesai
[ocr] 210/345 halaman selesai
[ocr] 220/345 halaman selesai
[ocr] 230/345 halaman selesai
[ocr] 240/345 halaman selesai
[ocr] 250/345 halaman selesai
[ocr] 260/345 halaman selesai
[ocr] 270/345 halaman selesai
[ocr] 280/345 halaman selesai
[ocr] 290/345 halaman selesai
[ocr] 300/345 halaman selesai
[ocr] 310/345 halaman selesai
[ocr] 320/345 halaman selesai
[ocr] 330/345 halaman selesai
[ocr] 340/345 halam

## Muat ulang hasil OCR dari checkpoint

Kalau kernel notebook sempat restart (misal setelah rename file), variabel `page_results` hilang dari memori — tapi datanya aman tersimpan di file. Sel ini mendefinisikan ulang path checkpoint dan memuat kembali hasil OCR yang sudah selesai, tanpa perlu OCR ulang.


In [12]:
CHECKPOINT_PATH = Path("data/ocr_raw/ocr_checkpoint.json")

In [13]:
with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
    page_results = json.load(f)

print(f"Total halaman ter-load: {len(page_results)}")

Total halaman ter-load: 345


## Tahap 3: Gabungkan teks halaman 1–228

Hasil OCR per halaman digabung jadi satu teks utuh, tapi **dibatasi sampai halaman 228 saja**. Berdasarkan pengecekan manual: Pasal 624 (pasal terakhir) selesai di halaman 228, halaman 229 berisi lembar pengesahan (tanda tangan Presiden, tanggal, dll — bukan pasal), dan halaman 230–345 adalah bagian **Penjelasan** UU (redaksi ulang nomor pasal yang sama, isinya kebanyakan cuma "Cukup jelas" — bukan teks hukum pasal itu sendiri). Kalau ikut disertakan, nanti chunking berdasarkan regex "Pasal N" akan menangkap tiap pasal 2x (dari batang tubuh dan dari Penjelasan).


In [14]:
BATAS_HALAMAN_PASAL = 228

pages_pasal = [p for p in page_results if p["page"] <= BATAS_HALAMAN_PASAL]
print(f"Jumlah halaman yang dipakai: {len(pages_pasal)}")

pages_pasal_sorted = sorted(pages_pasal, key=lambda x: x["page"])
full_text = "\n".join(p["text"] for p in pages_pasal_sorted)

print(f"Panjang teks gabungan: {len(full_text)} karakter")

Jumlah halaman yang dipakai: 228
Panjang teks gabungan: 355468 karakter


In [15]:
print(full_text[:10000])

Undang-Undang Republik Indonesia

NOMOR 1 TAHUN 2023

TENTANG

KITAB Undang-Undang HUKUM PIDANA

DENGAN RAHMAT TUHAN YANG MAHA ESA

PRESIDEN REPUBLIK INDONESIA,

Menimbang:

a. bahwa untuk mewujudkan hukum pidana nasional Negara Kesatuan Republik Indonesia yang berdasarkan Pancasila dan Undang-Undang Dasar Negara Republik Indonesia Tahun 1945 serta asas hukum umum yang diakui masyarakat bangsa-bangsa, perlu disusun hukum pidana nasional untuk mengganti Kitab Undang-Undang Hukum Pidana warisan pemerintah kolonial Hindia Belanda;

b. bahwa hukum pidana nasional tersebut harus disesuaikan dengan politik hukum, keadaan, dan perkembangan kehidupan bernasyarakat, berbangsa, dan bernegara yang bertujuan menghormati dan menjunjung tinggi hak asasi manusia, berdasarkan Ketuhanan Yang Maha Esa, kemanusiaan yang adil dan beradab, persatuan Indonesia, kerakyatan yang dipimpin oleh hikmat kebijaksanaan dalam permusyawaratan/perwakilan, dan keadilan sosial bagi seluruh rakyat Indonesia;

c. bahwa ma

## Tahap 4: Cleansing

Hasil OCR mentah (`full_text`) masih mengandung beberapa jenis noise yang 
berulang di tiap halaman, hasil temuan dari pengecekan manual sebelumnya:

1. **Penanda sambungan halaman** — pola seperti `"Pasal 2 . . ."` (dengan 
   titik-titik) muncul di akhir halaman sebagai penanda "pasal berikutnya 
   ada di halaman selanjutnya". Ini **bukan** awal pasal sungguhan — isi 
   pasal yang sebenarnya baru muncul di halaman berikutnya dengan format 
   `"Pasal 2"` (langsung diikuti isi ayat).
2. **Header berulang** — teks `"PRESIDEN REPUBLIK INDONESIA"` dan kode 
   `"SK No xxxxxx A"` muncul di tiap halaman sebagai kop dokumen resmi.
3. **Nomor halaman** — pola seperti `"- 3 -"`.

Ketiga jenis noise ini dibersihkan dengan regex, supaya nanti tidak 
mengganggu proses chunking per pasal (khususnya poin 1 — kalau tidak 
dibersihkan, regex chunking bisa salah mendeteksi "Pasal N . . ." sebagai 
awal chunk baru, padahal isinya kosong).

In [ ]:
import re

def clean_text(text: str) -> str:
    # 1. Hapus baris pendek yang jadi penanda sambungan halaman
    #    (mencakup "Pasal N . . .", "(5) Dalam . . .", "a. keamanan . . .",
    #    "Paragraf 4 . . .", "Bagian Keempat . . .", "d. bahwa...", dst)
    text = re.sub(
        r"(?m)^[ \t]*\S[^\n]{0,80}?\.\s*\.\s*\.\s*[ \t]*$\n?",
        "",
        text
    )
    
    # 2. Hapus header berulang
    text = re.sub(r"PRESIDEN\s*\n?REPUBLIK INDONESIA,?", "", text)
    text = re.sub(r"SK No \d+ [A-Z]", "", text)
    
    # 3. Hapus nomor halaman pola "- 3 -"
    text = re.sub(r"-\s*\d+\s*-", "", text)
    
    # 4. Rapikan baris kosong berlebih & spasi ganda
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    
    return text.strip()

full_text_clean = clean_text(full_text)
print(f"Panjang sebelum: {len(full_text)} karakter")
print(f"Panjang sesudah: {len(full_text_clean)} karakter")

Panjang sebelum: 355468 karakter
Panjang sesudah: 349900 karakter


In [59]:
print(full_text_clean[:100000])

Undang-Undang Republik Indonesia

NOMOR 1 TAHUN 2023

TENTANG

KITAB Undang-Undang HUKUM PIDANA

DENGAN RAHMAT TUHAN YANG MAHA ESA

Menimbang:

a. bahwa untuk mewujudkan hukum pidana nasional Negara Kesatuan Republik Indonesia yang berdasarkan Pancasila dan Undang-Undang Dasar Negara Republik Indonesia Tahun 1945 serta asas hukum umum yang diakui masyarakat bangsa-bangsa, perlu disusun hukum pidana nasional untuk mengganti Kitab Undang-Undang Hukum Pidana warisan pemerintah kolonial Hindia Belanda;

b. bahwa hukum pidana nasional tersebut harus disesuaikan dengan politik hukum, keadaan, dan perkembangan kehidupan bernasyarakat, berbangsa, dan bernegara yang bertujuan menghormati dan menjunjung tinggi hak asasi manusia, berdasarkan Ketuhanan Yang Maha Esa, kemanusiaan yang adil dan beradab, persatuan Indonesia, kerakyatan yang dipimpin oleh hikmat kebijaksanaan dalam permusyawaratan/perwakilan, dan keadilan sosial bagi seluruh rakyat Indonesia;

c. bahwa materi hukum pidana nasional jug

In [60]:
sisa = re.findall(r"[^\n]{0,60}\.\s*\.\s*\.\s*", full_text_clean)
print(f"Masih ada {len(sisa)} baris dengan pola titik-titik")
for s in sisa[:10]:
    print(repr(s))

Masih ada 0 baris dengan pola titik-titik


## Tahap 5: Chunking per pasal + metadata struktural (Buku/Bab/Bagian/Paragraf)

`full_text_clean` sekarang siap dipecah jadi 624 potongan per pasal. Pendekatannya: **satu kali baca teks baris demi baris**, sambil melacak posisi struktural yang sedang berjalan (Buku → Bab → Bagian → Paragraf). Setiap kali ketemu heading salah satu level itu, statusnya di-*update* (dan level di bawahnya di-reset ke `None`, karena masuk Bab/Bagian/Paragraf baru berarti yang lama sudah tidak berlaku lagi). Setiap kali ketemu baris `"Pasal N"`, chunk sebelumnya ditutup dan chunk baru dibuka — metadatanya langsung dicap dari status yang sedang berjalan saat itu juga, bukan dihitung ulang belakangan.

Fungsi yang dipakai:
- `parse_kuhp()` — parser utama, menghasilkan list `chunks` dengan field `pasal`, `buku`, `bab`, `bagian`, `paragraf`, `text`.
- `validate()` — cek nomor pasal berurutan 1–624 tanpa lompat/dobel, dan tidak ada chunk kosong.

Hasil jalan pertama kali di bawah masih menunjukkan beberapa anomali — dibedah di Tahap 6.


In [61]:
import re
import json

RE_BUKU = re.compile(r"^BUKU [A-Z]+$")
RE_BAB = re.compile(r"^BAB [IVXLCM]+$")
RE_BAGIAN = re.compile(r"^Bagian [A-Za-z]+$")
RE_PARAGRAF = re.compile(r"^Paragraf \d+$")
RE_PASAL = re.compile(r"^Pasal (\d+)$")

def is_heading(s: str) -> bool:
    return bool(
        RE_BUKU.match(s) or RE_BAB.match(s) or
        RE_BAGIAN.match(s) or RE_PARAGRAF.match(s) or RE_PASAL.match(s)
    )

def collect_title(lines, start_idx):
    """Lewati baris kosong di awal, lalu gabung baris non-kosong berikutnya
    sampai ketemu baris kosong atau heading/pasal lain (safety net)."""
    j = start_idx
    while j < len(lines) and lines[j].strip() == "":
        j += 1
    title_parts = []
    while j < len(lines) and lines[j].strip() != "" and not is_heading(lines[j].strip()):
        title_parts.append(lines[j].strip())
        j += 1
    return " ".join(title_parts), j


def parse_kuhp(full_text_clean: str):
    lines = full_text_clean.split("\n")
    n = len(lines)
    i = 0
    buku = bab = bagian = paragraf = None
    chunks = []
    current = None

    while i < n:
        s = lines[i].strip()

        if RE_BUKU.match(s):
            title, i = collect_title(lines, i + 1)
            buku = f"{s} - {title}" if title else s
            bab = bagian = paragraf = None
            continue

        if RE_BAB.match(s):
            title, i = collect_title(lines, i + 1)
            bab = f"{s} - {title}" if title else s
            bagian = paragraf = None
            continue

        if RE_BAGIAN.match(s):
            title, i = collect_title(lines, i + 1)
            bagian = f"{s} - {title}" if title else s
            paragraf = None
            continue

        if RE_PARAGRAF.match(s):
            title, i = collect_title(lines, i + 1)
            paragraf = f"{s} - {title}" if title else s
            continue

        m = RE_PASAL.match(s)
        if m:
            if current is not None:
                current["text"] = "\n".join(current["text_lines"]).strip()
                del current["text_lines"]
                chunks.append(current)
            current = {
                "pasal": int(m.group(1)),
                "buku": buku, "bab": bab,
                "bagian": bagian, "paragraf": paragraf,
                "text_lines": [lines[i]],
            }
            i += 1
            continue

        if current is not None:
            current["text_lines"].append(lines[i])
        i += 1

    if current is not None:
        current["text"] = "\n".join(current["text_lines"]).strip()
        del current["text_lines"]
        chunks.append(current)
    return chunks


def validate(chunks):
    nomor = [c["pasal"] for c in chunks]
    seen = set()
    duplikat = sorted(set(x for x in nomor if x in seen or seen.add(x)))
    urut = list(range(min(nomor), max(nomor) + 1))
    hilang = sorted(set(urut) - set(nomor))
    kosong = [c["pasal"] for c in chunks if not c["text"].strip()]
    return {"total_chunk": len(chunks), "pasal_unik": len(set(nomor)),
            "duplikat": duplikat, "hilang": hilang, "chunk_kosong": kosong}

In [62]:
chunks = parse_kuhp(full_text_clean)
print(f"Jumlah chunk terbentuk: {len(chunks)}")

report = validate(chunks)
print(f"Pasal unik: {report['pasal_unik']} dari {report['total_chunk']} chunk")
print(f"Masih ada {len(report['duplikat'])} pasal duplikat: {report['duplikat']}")
print(f"Masih ada {len(report['hilang'])} pasal hilang: {report['hilang']}")
print(f"Masih ada {len(report['chunk_kosong'])} chunk kosong: {report['chunk_kosong']}")

Jumlah chunk terbentuk: 625
Pasal unik: 620 dari 625 chunk
Masih ada 5 pasal duplikat: [93, 94, 95, 576, 577]
Masih ada 4 pasal hilang: [289, 448, 457, 620]
Masih ada 0 chunk kosong: []


In [63]:
contoh_nomor = [1, 4, 148, 203]
for c in chunks:
    if c["pasal"] in contoh_nomor:
        print(json.dumps({k: v for k, v in c.items() if k != "text"}, ensure_ascii=False, indent=2))
        print(c["text"][:120], "...\n")

{
  "pasal": 1,
  "buku": "BUKU KESATU - ATURAN UMUM",
  "bab": "BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PERATURAN PERUNDANG-UNDANGAN PIDANA",
  "bagian": "Bagian Kesatu - Menurut Waktu",
  "paragraf": null
}
Pasal 1
(1) Tidak ada satu perbuatan pun yang dapat dikenai sanksi pidana dan/atau tindakan, kecuali atas kekuatan perat ...

{
  "pasal": 4,
  "buku": "BUKU KESATU - ATURAN UMUM",
  "bab": "BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PERATURAN PERUNDANG-UNDANGAN PIDANA",
  "bagian": "Bagian Kedua - Menurut Tempat",
  "paragraf": "Paragraf 1 - Asas Wilayah atau Teritorial"
}
Pasal 4
Ketentuan pidana dalam Undang-Undang berlaku bagi Setiap Orang yang melakukan:
a. Tindak Pidana di wilayah Negar ...

{
  "pasal": 148,
  "buku": "BUKU KESATU - ATURAN UMUM",
  "bab": "BAB V - PENGERTIAN ISTILAH",
  "bagian": null,
  "paragraf": null
}
Pasal 148
Surat adalah dokumen yang ditulis di atas kertas, termasuk juga dokumen atau data yang tertulis atau tersimpan ...

{
  "pasal": 203,
  "buk

## Tahap 6: Validasi & debugging anomali

Hasil `validate()` di atas menunjukkan dua masalah terpisah:
- **5 pasal duplikat**: 93, 94, 95, 576, 577
- **4 pasal hilang**: 289, 448, 457, 620

Sel-sel di bawah ini menelusuri akar penyebab masing-masing sebelum ditambal, supaya perbaikannya tertarget (bukan asal tebak):

- Cek apakah duplikat berasal dari **halaman OCR yang ke-proses dua kali** (`dupe_halaman` — ternyata kosong, jadi bukan ini penyebabnya).
- Intip isi tiap versi pasal yang duplikat, untuk lihat apakah isinya identik atau beda.
- Cari lokasi pasal yang hilang di `full_text_clean`, pakai pasal sebelumnya sebagai patokan baris.
- Cek di halaman berapa tiap nomor duplikat muncul sebagai heading — ternyata masing-masing cuma muncul di **satu** halaman, bukan dua halaman berbeda. Artinya OCR-nya sendiri yang mengulang balok teks di satu halaman, bukan halaman yang digabung dua kali.
- Cek versi **mentah** (`full_text`, sebelum `clean_text()`) di sekitar lokasi pasal yang hilang — ketemu pola `"Pasal 289 . . ."`, `"Pasal 448 . . ."`, `"Pasal 620 . . ."` (heading yang salah tertulis dengan titik-titik, lalu terhapus habis oleh regex pembersih di Tahap 4), dan satu kasus lain (`"BAB XX . . ."` menggantikan "Pasal 457") yang OCR-nya salah baca total, bukan cuma soal titik-titik.


In [64]:
from collections import Counter

halaman_count = Counter(p["page"] for p in pages_pasal)
dupe_halaman = [h for h, jml in halaman_count.items() if jml > 1]
print(f"Halaman yang muncul lebih dari sekali di pages_pasal: {dupe_halaman}")

Halaman yang muncul lebih dari sekali di pages_pasal: []


In [65]:
cnt = Counter(c["pasal"] for c in chunks)
for n in sorted(k for k, v in cnt.items() if v > 1):
    versi = [c for c in chunks if c["pasal"] == n]
    print(f"=== Pasal {n} ({len(versi)} versi) ===")
    for idx, v in enumerate(versi, 1):
        print(f"--versi {idx}--")
        print(v["text"][:150])
    print()

=== Pasal 93 (2 versi) ===
--versi 1--
Pasal 93

(1) Jika dalam putusan pengadilan diperintahkan supaya putusan diumumkan, harus ditetapkan cara melaksanakan pengumuman tersebut dengan biay
--versi 2--
Pasal 93

(1) Jika dalam putusan pengadilan diperintahkan supaya putusan diumumkan, harus ditetapkan cara melaksanakan pengumuman tersebut dengan biay

=== Pasal 94 (2 versi) ===
--versi 1--
Pasal 94

(1) Dalam putusan pengadilan dapat ditetapkan kewajiban terpidana untuk melaksanakan pembayaran ganti rugi kepada Korban atau ahli waris seb
--versi 2--
Pasal 94

(1) Dalam putusan pengadilan dapat ditetapkan kewajiban terpidana untuk melaksanakan pembayaran ganti rugi kepada Korban atau ahli waris seb

=== Pasal 95 (2 versi) ===
--versi 1--
Pasal 95

(1) Pidana tambahan berupa pencabutan izin dikenakan kepada pelaku dan pembantu Tindak Pidana yang melakukan Tindak Pidana yang berkaitan de
--versi 2--
Pasal 95

(1) Pidana tambahan berupa pencabutan izin dikenakan kepada pelaku dan pembantu 

In [66]:
def cari_konteks_hilang(nomor, full_text_clean, maks_baris=40):
    lines = full_text_clean.split("\n")
    target = f"Pasal {nomor - 1}"
    for i, line in enumerate(lines):
        if line.strip() == target:
            print(f"=== sekitar Pasal {nomor} (patokan: Pasal {nomor - 1} di baris {i}) ===")
            for j in range(i, min(i + maks_baris, len(lines))):
                print(j, repr(lines[j]))
            print()
            return
    print(f"Pasal {nomor - 1} juga tidak ketemu sebagai baris tersendiri.")

for n in [289, 448, 457, 620]:
    cari_konteks_hilang(n, full_text_clean)

=== sekitar Pasal 289 (patokan: Pasal 288 di baris 2299) ===
2299 'Pasal 288'
2300 ''
2301 'Setiap Orang yang tanpa alasan yang sah tidak datang menghadap atau dalam hal yang diizinkan tidak meminta wakilnya menghadap, jika dipanggil di muka pengadilan untuk didengar sebagai keluarga sedarah atau keluarga semenda, suami atau istri, wali atau wali pengawas, pengampu atau pengampu pengawas dalam perkara orang yang akan ditaruh atau yang sudah ditaruh di bawah pengampuan atau dalam perkara orang yang akan dimasukkan atau sudah dimasukkan ke rumah sakit jiwa, dipidana dengan pidana denda paling banyak kategori II.'
2302 ''
2303 '(1) Dipidana dengan pidana penjara paling lama 4 (empat) tahun atau pidana denda paling banyak kategori V, Setiap Orang yang:'
2304 ''
2305 'a. menarik Barang yang disita berdasarkan peraturan perundang-undangan atau yang dititipkan atas perintah pengadilan atau menyembunyikan Barang, padahal diketahui bahwa Barang tersebut berada dalam sitaan atau titipan; atau'
2

In [67]:
for nomor in [93, 94, 95, 576, 577]:
    halaman = [p["page"] for p in pages_pasal if re.search(rf"(?m)^Pasal {nomor}\s*$", p["text"])]
    print(f"Pasal {nomor} muncul sebagai heading di halaman: {halaman}")

Pasal 93 muncul sebagai heading di halaman: [36]
Pasal 94 muncul sebagai heading di halaman: [36]
Pasal 95 muncul sebagai heading di halaman: [36]
Pasal 576 muncul sebagai heading di halaman: [198]
Pasal 577 muncul sebagai heading di halaman: [198]


In [68]:
def cari_mentah(nomor, full_text, maks_baris=15):
    lines = full_text.split("\n")
    target = f"Pasal {nomor - 1}"
    idx = [i for i, l in enumerate(lines) if l.strip().startswith(target)]
    for i in idx:
        print(f"=== mentah, sekitar Pasal {nomor} (baris {i}) ===")
        for j in range(i, min(i + maks_baris, len(lines))):
            print(j, repr(lines[j]))
        print()

for n in [289, 448, 457, 620]:
    cari_mentah(n, full_text)

=== mentah, sekitar Pasal 289 (baris 2435) ===
2435 'Pasal 288'
2436 ''
2437 'Setiap Orang yang tanpa alasan yang sah tidak datang menghadap atau dalam hal yang diizinkan tidak meminta wakilnya menghadap, jika dipanggil di muka pengadilan untuk didengar sebagai keluarga sedarah atau keluarga semenda, suami atau istri, wali atau wali pengawas, pengampu atau pengampu pengawas dalam perkara orang yang akan ditaruh atau yang sudah ditaruh di bawah pengampuan atau dalam perkara orang yang akan dimasukkan atau sudah dimasukkan ke rumah sakit jiwa, dipidana dengan pidana denda paling banyak kategori II.'
2438 ''
2439 'Pasal 289 . . .'
2440 '(1) Dipidana dengan pidana penjara paling lama 4 (empat) tahun atau pidana denda paling banyak kategori V, Setiap Orang yang:'
2441 ''
2442 'a. menarik Barang yang disita berdasarkan peraturan perundang-undangan atau yang dititipkan atas perintah pengadilan atau menyembunyikan Barang, padahal diketahui bahwa Barang tersebut berada dalam sitaan atau titipan

## Tahap 7: Perbaikan tertarget

Berdasarkan investigasi di atas, perbaikannya **tertarget ke nomor pasal spesifik**, bukan regex umum yang berlaku ke seluruh dokumen (regex umum sempat dicoba dan malah merusak 72 pasal lain yang sebenarnya sudah benar — pelajaran: pola `"Pasal N . . ."` itu biasanya memang preview halaman yang harus dibuang, cuma di 3 kasus ini kebetulan itu satu-satunya heading yang ada untuk nomor itu).

- `dedup_pasal()` — untuk 5 pasal duplikat: kalau isi kedua versi identik, ambil salah satu; kalau beda, ambil versi dengan teks **terpanjang** (setelah dicek manual di Tahap 8, semua kasus beda ternyata murni "versi pendek terpotong di tengah", bukan konten yang bertentangan).
- Patch 289, 448, 620 — tambal `full_text` mentah di 3 titik spesifik itu saja (`assert count == 1` untuk mastikan tidak salah tempel), baru panggil ulang `clean_text()`.
- Patch 457 — kasus khusus, heading-nya salah OCR total, ditambal manual pakai kalimat pertama isi pasalnya sebagai jangkar.


In [90]:
from collections import defaultdict

def dedup_pasal(chunks):
    by_pasal = defaultdict(list)
    for c in chunks:
        by_pasal[c["pasal"]].append(c)

    hasil, perlu_dicek = [], []
    for nomor, versi in sorted(by_pasal.items()):
        if len(versi) == 1:
            hasil.append(versi[0])
            continue
        teks_unik = {v["text"] for v in versi}
        if len(teks_unik) == 1:
            hasil.append(versi[0])
        else:
            terpanjang = max(versi, key=lambda v: len(v["text"]))
            hasil.append(terpanjang)
            perlu_dicek.append(nomor)
    return hasil, perlu_dicek

In [91]:
perbaikan_tertarget = {
    289: "Pasal 289 . . .",
    448: "Pasal 448 . . .",
    620: "Pasal 620 . . .",
}

full_text_fix = full_text
for nomor, pola_asli in perbaikan_tertarget.items():
    jumlah = full_text_fix.count(pola_asli)
    assert jumlah == 1, f"Pasal {nomor}: pola ditemukan {jumlah}x, seharusnya 1x"
    full_text_fix = full_text_fix.replace(pola_asli, f"Pasal {nomor}")

full_text_clean = clean_text(full_text_fix)
print(f"Panjang sesudah: {len(full_text_clean)} karakter")

Panjang sesudah: 349932 karakter


In [92]:
anchor = "Setiap Orang yang melakukan perbuatan yang bertujuan mencari keuntungan, baik secara langsung maupun tidak langsung, untuk diri sendiri atau untuk orang lain dengan membawa seseorang"
assert full_text_clean.count(anchor) == 1, "anchor tidak unik atau tidak ketemu"
full_text_clean = full_text_clean.replace(anchor, "Pasal 457\n\n" + anchor)
print("Pasal 457 ditambal")

Pasal 457 ditambal


## Tahap 8: Chunking ulang & validasi setelah perbaikan

`full_text_clean` sudah diperbarui dengan 3 patch tertarget + patch Pasal 457 di atas. Chunking dan validasi dijalankan ulang dari sini:

1. `parse_kuhp()` dipanggil ulang dengan `full_text_clean` yang baru.
2. `dedup_pasal()` dipanggil untuk menyatukan 5 pasal yang masih dobel.
3. `validate()` dipanggil pada `chunks_final` — target: 624 pasal unik, 0 duplikat, 0 hilang, 0 chunk kosong.
4. Verifikasi tambahan untuk 3 pasal yang butuh keputusan "pilih versi terpanjang" (95, 576, 577): cek diff detail antar versinya, panjang teks di `chunks_final`, dan pastikan frasa yang tadinya cuma ada di versi panjang sekarang benar-benar ikut ke chunk final — supaya yakin bukan cuma lolos hitungan, tapi isinya juga lengkap.


> **Kenapa hasil chunking pertama (Tahap 5) tidak dihapus:** itu bukan sisa yang kelupaan, itu bukti diagnosis. Tanpa hasil pertama yang menunjukkan 5 pasal duplikat + 4 pasal hilang, patch-patch di Tahap 7 (289/448/620/457, dan logika "pilih versi terpanjang" di `dedup_pasal`) akan kelihatan seperti ditebak asal — padahal semuanya berdasarkan temuan nyata dari data. Notebook ini sengaja disusun supaya proses debugging-nya kelihatan, bukan cuma hasil akhir yang sudah mulus.


In [103]:
chunks = parse_kuhp(full_text_clean)
print(f"Jumlah chunk terbentuk: {len(chunks)}")

Jumlah chunk terbentuk: 629


In [94]:
chunks_final, perlu_dicek = dedup_pasal(chunks)
print(f"Jumlah chunk setelah dedup: {len(chunks_final)}")
print(f"Perlu dicek manual: {perlu_dicek}")

Jumlah chunk setelah dedup: 624
Perlu dicek manual: [95, 576, 577]


In [95]:

report = validate(chunks_final)
print(f"Pasal unik: {report['pasal_unik']} dari {report['total_chunk']} chunk")
print(f"Masih ada {len(report['duplikat'])} pasal duplikat: {report['duplikat']}")
print(f"Masih ada {len(report['hilang'])} pasal hilang: {report['hilang']}")
print(f"Masih ada {len(report['chunk_kosong'])} chunk kosong: {report['chunk_kosong']}")

Pasal unik: 624 dari 624 chunk
Masih ada 0 pasal duplikat: []
Masih ada 0 pasal hilang: []
Masih ada 0 chunk kosong: []


In [97]:
import difflib

for n in perlu_dicek:
    versi = [c["text"] for c in chunks if c["pasal"] == n]
    print(f"=== Pasal {n} ({len(versi)} versi) ===")
    for i in range(1, len(versi)):
        diff = difflib.unified_diff(
            versi[0].split(), versi[i].split(),
            lineterm="", fromfile="versi 1", tofile=f"versi {i+1}"
        )
        print("\n".join(diff))
    print()

=== Pasal 95 (2 versi) ===
--- versi 1
+++ versi 2
@@ -97,3 +97,38 @@
 pokok
 yang
 dijatuhkan.
+(4)
+Dalam
+hal
+dijatuhi
+pidana
+denda,
+pencabutan
+izin
+berlaku
+paling
+singkat
+1
+(satu)
+tahun
+dan
+paling
+lama
+5
+(lima)
+tahun.
+(5)
+Pidana
+pencabutan
+izin
+mulai
+berlaku
+pada
+tanggal
+putusan
+pengadilan
+telah
+memperoleh
+kekuatan
+hukum
+tetap.

=== Pasal 576 (2 versi) ===
--- versi 1
+++ versi 2
@@ -18,14 +18,7 @@
 pengamanan
 lalu
 lintas
-udara
-atau
-gagalnya
-usaha
-untuk
-pengamanan
-bangunan
-tersebut,
+udara,
 dipidana
 dengan
 pidana

=== Pasal 577 (2 versi) ===
--- versi 1
+++ versi 2
@@ -108,50 +108,3 @@
 (lima
 belas)
 tahun.
-(2)
-Jika
-Tindak
-Pidana
-sebagaimana
-dimaksud
-pada
-ayat
-(1)
-menimbulkan
-bahaya
-bagi
-keamanan
-lalu
-lintas
-udara,
-dipidana
-dengan
-pidana
-penjara
-paling
-lama
-9
-(sembilan)
-tahun.
-(3)
-Jika
-Tindak
-Pidana
-sebagaimana
-dimaksud
-pada
-ayat
-(1)
-mengakibatkan
-matinya
-orang,
-dipidana
-dengan
-pidana
-penjara
-pa

In [98]:
for n in [95, 576, 577]:
    c = next(x for x in chunks_final if x["pasal"] == n)
    print(f"Pasal {n}: panjang text = {len(c['text'])} karakter")

Pasal 95: panjang text = 934 karakter
Pasal 576: panjang text = 584 karakter
Pasal 577: panjang text = 1117 karakter


In [99]:
checks = {
    95: "Pidana pencabutan izin mulai berlaku",
    576: "gagalnya usaha",
    577: "menimbulkan bahaya bagi keamanan lalu lintas udara",
}
for n, frasa in checks.items():
    c = next(x for x in chunks_final if x["pasal"] == n)
    print(f"Pasal {n}: {'lengkap ✅' if frasa in c['text'] else 'MASIH TERPOTONG ❌'}")

Pasal 95: lengkap ✅
Pasal 576: lengkap ✅
Pasal 577: lengkap ✅


## Tahap 9: Simpan hasil final ke JSON

Setelah semua validasi di atas bersih (624/624, tidak ada duplikat, tidak ada yang hilang, dan isi tiga pasal yang sempat rawan terpotong sudah dikonfirmasi lengkap), `chunks_final` disimpan sebagai dataset final ke `data/processed/chunks.json` — siap dipakai untuk tahap embedding (BGE-M3) dan pembuatan index FAISS.


In [101]:

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_final, f, ensure_ascii=False, indent=2)